<a href="https://colab.research.google.com/github/op1154/MBA_Agents/blob/main/Aula8_MBA_Memoria_Agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Memória dos Agentes
# **Short Term com Checkpoint***

In [31]:
from langchain.chat_models import init_chat_model
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.checkpoint.memory import InMemorySaver


In [32]:
!pip install langchain-openai -q

In [33]:
import os
from google.colab import userdata
admin_api_key = userdata.get('OPENAI_API_KEY')
os.environ["OPENAI_API_KEY"] = admin_api_key

model = init_chat_model("openai:gpt-4o-mini")



def tutor(state: MessagesState):

    return {"messages": [model.invoke(state["messages"])]}



builder = StateGraph(MessagesState)

builder.add_node("tutor", tutor)

builder.add_edge(START, "tutor")



graph = builder.compile(checkpointer=InMemorySaver())





In [34]:

cfg = {"configurable": {"thread_id": "aluno-1"}}

graph.invoke({"messages": [("user", "Oi, sou a Ana")]}, cfg)

out = graph.invoke({"messages": [("user", "Qual é o meu nome?")]}, cfg)

print(out["messages"][-1].content)   # "Esperado: Seu nome é Ana."

Seu nome é Ana. Como posso ajudá-la hoje?


In [35]:

cfg = {"configurable": {"thread_id": "aluno-2"}}

graph.invoke({"messages": [("user", "Eu sou o Carlos")]}, cfg)

out = graph.invoke({"messages": [("user", "Qual é o meu nome?")]}, cfg)

print(out["messages"][-1].content)   # "Esperado: Carlos."

Seu nome é Carlos. Como posso ajudar você hoje?


In [36]:
cfg = {"configurable": {"thread_id": "aluno-2"}}
# inspecionar e voltar no tempo​

snap = graph.get_state(cfg)

hist = list(graph.get_state_history(cfg))

for index, state in enumerate(hist):
    print(f"--- Estado {index} ---")
    print(f"Config: {state.config}")
    print(f"Valores: {state.values}")
    print(f"Próximo nó: {state.next}")
    print("-" * 40)


--- Estado 0 ---
Config: {'configurable': {'thread_id': 'aluno-2', 'checkpoint_ns': '', 'checkpoint_id': '1f1c28d6-aecb-64ec-8004-297e4de4820c'}}
Valores: {'messages': [HumanMessage(content='Eu sou o Carlos', additional_kwargs={}, response_metadata={}, id='5ee021cd-608f-428b-8e79-dbaa244639e2'), AIMessage(content='Olá, Carlos! Como posso te ajudar hoje?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 11, 'total_tokens': 21, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_551db23bc9', 'id': 'chatcmpl-EWSVoiMiJSe8TXZprbrljb5mCu09T', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1180c-05ff-7421-b369-2f6419d3

In [37]:
from langchain_core.messages import RemoveMessage
from langchain_core.messages.utils import (
trim_messages, count_tokens_approximately)



# DELETE: limpa o estado salvo​
def limpar(state: MessagesState):
    antigas = state["messages"][:-10]

    return {"messages": [RemoveMessage(id=m.id)

                         for m in antigas]}

In [38]:
estado_atual = graph.get_state(cfg)
print("=== Configuração Atual ===")
print(estado_atual.config)
print("\n=== Mensagens no Estado Atual ===")
for msg in estado_atual.values.get("messages", []):
    print(f"{msg.type.upper()}: {msg.content}")

=== Configuração Atual ===
{'configurable': {'thread_id': 'aluno-2', 'checkpoint_ns': '', 'checkpoint_id': '1f1c28d6-aecb-64ec-8004-297e4de4820c'}}

=== Mensagens no Estado Atual ===
HUMAN: Eu sou o Carlos
AI: Olá, Carlos! Como posso te ajudar hoje?
HUMAN: Qual é o meu nome?
AI: Seu nome é Carlos. Como posso ajudar você hoje?


Memória dos Agentes
# **Long Term com Gravação**

In [39]:
import uuid
from langchain_core.runnables import RunnableConfig
from langgraph.store.base import BaseStore
from langgraph.store.memory import InMemoryStore



store = InMemoryStore(index={"embed": "openai:text-embedding-3-small", "dims": 1536})



def tutormem(state: MessagesState, config: RunnableConfig, *, store: BaseStore):

  ns = ("memories", config["configurable"]["user_id"])

  pergunta = state["messages"][-1].content

  fatos = store.search(ns, query=pergunta, limit=3)   # LER​

  perfil = "\n".join(f.value["data"] for f in fatos)

  if "lembre" in pergunta.lower():                    # ESCREVER​

    store.put(ns, str(uuid.uuid4()), {"data": pergunta})

    sys = {"role": "system", "content": f"Perfil do aluno:\n{perfil}"}

    return {"messages": [model.invoke([sys, *state["messages"]])]}


graph = builder.compile(checkpointer=InMemorySaver(), store=store)

cfg = {"configurable": {"thread_id": "t1", "user_id": "ana"}}

In [40]:
# 1. Definindo o namespace que foi utilizado para salvar os dados
namespace = ("memories", "ana")

In [41]:


# 2. Recuperando todos os registros ou buscando por similaridade semântica usando search
resultados_busca = store.search(namespace, query="Qual o nome?", limit=5)
print("=== Resultados da Busca (search) ===")
for item in resultados_busca:
    print(f"ID: {item.key} | Conteúdo: {item.value}")

# 3. Caso você saiba o ID exato (key) de um item, você pode recuperá-lo diretamente usando get
# Exemplo (substitua pelo ID correto se houver):
#item = store.get(namespace, key="id-da-memoria")
#if item:
#  print(item.value)


=== Resultados da Busca (search) ===


In [42]:
# Insere um dado passando (namespace, chave_única, dicionário_com_os_dados)
store.put(("memories", "ana"), "user_preference", {"data": "A Ana gosta de cafe sem açucar"})

Memória dos Agentes
# **Long Term com Gravação em Banco**

In [51]:
import uuid

# 1. Em vez de usar uma chave fixa como "user_preference", geramos chaves únicas (UUID)
chave_1 = str(uuid.uuid4())
chave_2 = str(uuid.uuid4())

# 2. Salvando múltiplas informações no mesmo namespace
store.put(("memories", "ana"), chave_1, {"data": "A Ana prefere estudar no período da manhã."})
store.put(("memories", "ana"), chave_2, {"data": "A Ana está estudando para a prova de Matemática."})

# 3. Agora, ao buscar, você verá todos os registros ativos neste namespace!
resultados = store.search(("memories", "ana"), query="Estudo", limit=10)
print(f"=== Total de registros encontrados: {len(resultados)} ===\n")
for item in resultados:
    print(f"Chave: {item.key} -> {item.value}")


=== Total de registros encontrados: 6 ===

Chave: 48f0518f-5c86-43f8-ac60-af8a44ca9c89 -> {'data': 'A Ana está estudando para a prova de Matemática.'}
Chave: 67c23afc-a717-4c9a-a467-813e6575c7a0 -> {'data': 'A Ana está estudando para a prova de Matemática.'}
Chave: bd181af5-30bd-4727-b72e-8cfdaefce812 -> {'data': 'A Ana está estudando para a prova de Matemática.'}
Chave: c3cc1cb7-dbb1-4f7b-8faa-da18d26aa897 -> {'data': 'A Ana prefere estudar no período da manhã.'}
Chave: dc724e42-383e-442b-9855-0e5b202815e2 -> {'data': 'A Ana prefere estudar no período da manhã.'}
Chave: 56290467-773a-4512-9b4c-0add85971fb5 -> {'data': 'A Ana prefere estudar no período da manhã.'}


In [48]:
# 1. Removemos a chave_1 (atribuindo None)
store.put(("memories", "ana"), "chave_1", None)

# 2. Tentamos buscar o registro removido
record = store.get(("memories", "ana"), "chave_1")

# 3. Verificação de segurança para evitar o erro AttributeError
if record is not None:
    print(record.value)
else:
    print("O registro para 'chave_1' foi deletado e agora é None.")

O registro para 'chave_1' foi deletado e agora é None.


In [49]:
# 1. Para deletar um registro específico, passamos None no valor (value)
store.put(("memories", "ana"), "user_preference", None)

# 2. Vamos buscar novamente para comprovar que o registro 'user_preference' foi removido
resultados_restantes = store.search(("memories", "ana"), query="cafe", limit=5)
print("=== Registros restantes após a deleção ===")
for item in resultados_restantes:
    print(f"Chave: {item.key} -> {item.value}")

=== Registros restantes após a deleção ===
Chave: dc724e42-383e-442b-9855-0e5b202815e2 -> {'data': 'A Ana prefere estudar no período da manhã.'}
Chave: c3cc1cb7-dbb1-4f7b-8faa-da18d26aa897 -> {'data': 'A Ana prefere estudar no período da manhã.'}
Chave: 48f0518f-5c86-43f8-ac60-af8a44ca9c89 -> {'data': 'A Ana está estudando para a prova de Matemática.'}
Chave: 67c23afc-a717-4c9a-a467-813e6575c7a0 -> {'data': 'A Ana está estudando para a prova de Matemática.'}


In [50]:
# Busca por similaridade aproximada se houver um indexador de embeddings ativo
resultados = store.search(("memories", "ana"), query="O que a Ana bebe?", limit=1)
for item in resultados:
    print(item.key, "->", item.value)

48f0518f-5c86-43f8-ac60-af8a44ca9c89 -> {'data': 'A Ana está estudando para a prova de Matemática.'}
67c23afc-a717-4c9a-a467-813e6575c7a0 -> {'data': 'A Ana está estudando para a prova de Matemática.'}
c3cc1cb7-dbb1-4f7b-8faa-da18d26aa897 -> {'data': 'A Ana prefere estudar no período da manhã.'}
dc724e42-383e-442b-9855-0e5b202815e2 -> {'data': 'A Ana prefere estudar no período da manhã.'}
